In [ ]:
import sys
from pathlib import Path

# ensure scripts/ is on sys.path so `import train` works from any cwd
_scripts_dir = str(Path(__file__).resolve().parent) if "__file__" in dir() else str(Path.cwd())
if _scripts_dir not in sys.path:
    sys.path.insert(0, _scripts_dir)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from train import (
    F, DEVICE, USE_AMP, PERIODS_PER_YEAR, REBAL_INTERVAL, SPLITS,
    set_seed, load_store, compute_rebal_indices,
    CryptoDataset, MultiBranchLSTM, evaluate, train, TrainConfig,
    collate_fn,
)

set_seed(42)
print(f"Device: {DEVICE}, AMP: {USE_AMP}")

## Load Data & Build Datasets

In [ ]:
SYMBOL = "BTCUSDT"
HIDDEN = 64
COST_BPS = 2.0
CHUNK_SIZE = 192

feature_dir = Path.home() / "Data" / "binance" / "features"
kline_root = Path.home() / "Data" / "binance"
ckpt_dir = Path.home() / "CryptoDL" / "checkpoints"

store = load_store(feature_dir, kline_root, SYMBOL)

print("\nComputing rebalance indices...")
split_indices = {}
for split_name in SPLITS:
    split_indices[split_name] = compute_rebal_indices(store, split_name)

datasets = {name: CryptoDataset(store, idx) for name, idx in split_indices.items()}

## Train (or skip if checkpoint exists)

In [ ]:
CKPT_PATH = ckpt_dir / f"{SYMBOL}_best_sharpe.pt"
TRAIN_NEW = False  # set True to retrain from scratch

model = MultiBranchLSTM(f=F, hidden=HIDDEN).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model: MultiBranchLSTM (hidden={HIDDEN}), {n_params:,} params")

if TRAIN_NEW or not CKPT_PATH.exists():
    print("Training from scratch...")
    cfg = TrainConfig(symbol=SYMBOL, hidden=HIDDEN, chunk_size=CHUNK_SIZE, cost_bps=COST_BPS)
    cfg.out_dir.mkdir(parents=True, exist_ok=True)
    best_sharpe = train(model, datasets["train"], datasets["val"], cfg)
    print(f"Best val Sharpe: {best_sharpe:.2f}")
else:
    state = torch.load(CKPT_PATH, weights_only=True, map_location=DEVICE)
    model.load_state_dict(state)
    print(f"Loaded checkpoint: {CKPT_PATH.name}")

## Evaluate All Splits

In [ ]:
from torch.utils.data import DataLoader

def evaluate_detailed(model, ds, indices, store, cost_bps=2.0, chunk_size=192):
    """Evaluate and return per-step positions, net_pnl, timestamps."""
    model.eval()
    loader = DataLoader(ds, batch_size=chunk_size, shuffle=False, collate_fn=collate_fn, num_workers=0)

    all_net_pnl, all_positions, all_gross_pnl = [], [], []
    prev_pos = torch.zeros(1, device=DEVICE)

    with torch.no_grad():
        for x8h, x1h, x1m, ret_bps, has_nan in loader:
            x8h = x8h.to(DEVICE, non_blocking=True)
            x1h = x1h.to(DEVICE, non_blocking=True)
            x1m = x1m.to(DEVICE, non_blocking=True)
            ret_bps = ret_bps.to(DEVICE, non_blocking=True)

            with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
                positions = model(x8h, x1h, x1m)
            positions = positions.float()

            gross = positions * ret_bps
            cat_pos = torch.cat([prev_pos, positions])
            turnover = torch.abs(cat_pos[1:] - cat_pos[:-1])
            net = gross - cost_bps * turnover

            all_net_pnl.append(net.cpu())
            all_gross_pnl.append(gross.cpu())
            all_positions.append(positions.cpu())
            prev_pos = positions[-1:].detach()

    net_pnl = torch.cat(all_net_pnl).numpy()
    gross_pnl = torch.cat(all_gross_pnl).numpy()
    positions = torch.cat(all_positions).numpy()
    timestamps = pd.to_datetime(store.ts_1m[indices], unit="s", utc=True)

    return {
        "net_pnl": net_pnl,
        "gross_pnl": gross_pnl,
        "positions": positions,
        "timestamps": timestamps,
    }

results = {}
for split_name in ["val", "test", "paper"]:
    ds = datasets[split_name]
    idx = split_indices[split_name]
    if len(idx) == 0:
        print(f"  {split_name}: no data, skipping")
        continue

    metrics = evaluate(model, ds, COST_BPS, CHUNK_SIZE)
    detailed = evaluate_detailed(model, ds, idx, store, COST_BPS, CHUNK_SIZE)
    results[split_name] = {**metrics, **detailed}

    # always-long baseline
    rets = []
    for i in idx:
        c0, c1 = store.close_1m[i], store.close_1m[i + REBAL_INTERVAL]
        if c0 > 0:
            rets.append(np.log(c1 / c0) * 10000)
    rets = np.array(rets)
    long_sharpe = (rets.mean() / rets.std()) * np.sqrt(PERIODS_PER_YEAR)
    results[split_name]["always_long_sharpe"] = long_sharpe
    results[split_name]["always_long_pnl"] = rets.sum()
    results[split_name]["always_long_cum"] = np.cumsum(rets)

# Summary table
rows = []
for name in ["val", "test", "paper"]:
    if name not in results:
        continue
    r = results[name]
    rows.append({
        "Split": name,
        "Sharpe": f"{r['sharpe_annual']:+.2f}",
        "PnL (bps)": f"{r['total_pnl_bps']:+,.0f}",
        "MaxDD (bps)": f"{r['max_drawdown_bps']:,.0f}",
        "Daily Turn": f"{r['daily_turnover']:.1f}",
        "|Pos|": f"{r['avg_abs_position']:.2f}",
        "% Long": f"{r['pct_long']:.1f}",
        "% Short": f"{r['pct_short']:.1f}",
        "% Flat": f"{r['pct_flat']:.1f}",
        "Long Sharpe": f"{r['always_long_sharpe']:+.2f}",
        "N": f"{r['n_periods']:,}",
    })
pd.DataFrame(rows).set_index("Split")

## Cumulative PnL Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, split_name in zip(axes, ["val", "test", "paper"]):
    if split_name not in results:
        ax.set_title(f"{split_name} (no data)")
        continue
    r = results[split_name]
    ts = r["timestamps"]
    cum_net = np.cumsum(r["net_pnl"])
    cum_gross = np.cumsum(r["gross_pnl"])
    cum_long = r["always_long_cum"]

    ax.plot(ts, cum_net, label=f"Model net (Sharpe={r['sharpe_annual']:+.2f})", linewidth=1.2)
    ax.plot(ts, cum_gross, label="Model gross", linewidth=0.8, alpha=0.5)
    ax.plot(ts, cum_long, label=f"Always-long (Sharpe={r['always_long_sharpe']:+.2f})",
            linewidth=0.8, linestyle="--", color="gray")
    ax.axhline(0, color="black", linewidth=0.3)
    ax.set_title(f"{split_name.upper()} — {SPLITS[split_name][0]} to {SPLITS[split_name][1]}")
    ax.set_ylabel("Cumulative PnL (bps)")
    ax.legend(fontsize=8)
    ax.tick_params(axis="x", rotation=30)

fig.suptitle(f"{SYMBOL} MultiBranchLSTM h={HIDDEN}, cost={COST_BPS}bps", fontsize=13)
fig.tight_layout()
plt.show()

## Position Distribution & Turnover

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8))

for col, split_name in enumerate(["val", "test", "paper"]):
    if split_name not in results:
        continue
    r = results[split_name]
    pos = r["positions"]
    ts = r["timestamps"]

    # Position over time
    ax = axes[0, col]
    ax.plot(ts, pos, linewidth=0.3, alpha=0.8)
    ax.axhline(0, color="black", linewidth=0.3)
    ax.set_ylim(-1.1, 1.1)
    ax.set_title(f"{split_name.upper()} — Position over time")
    ax.set_ylabel("Position")
    ax.tick_params(axis="x", rotation=30)

    # Position histogram
    ax = axes[1, col]
    ax.hist(pos, bins=100, edgecolor="none", alpha=0.8)
    ax.axvline(0, color="black", linewidth=0.5)
    ax.set_title(f"{split_name.upper()} — Position distribution")
    ax.set_xlabel("Position")
    ax.set_ylabel("Count")

    turnover = np.abs(np.diff(pos))
    stats_text = (
        f"|pos| mean={np.abs(pos).mean():.2f}\n"
        f"turnover/rebal={turnover.mean():.4f}\n"
        f"daily turn={turnover.mean()*96:.1f}"
    )
    ax.text(0.02, 0.95, stats_text, transform=ax.transAxes, fontsize=8,
            verticalalignment="top", fontfamily="monospace",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

fig.suptitle(f"{SYMBOL} MultiBranchLSTM h={HIDDEN}", fontsize=13)
fig.tight_layout()
plt.show()

## Monthly PnL Breakdown

In [ ]:
for split_name in ["val", "test", "paper"]:
    if split_name not in results:
        continue
    r = results[split_name]
    df = pd.DataFrame({
        "timestamp": r["timestamps"],
        "net_pnl": r["net_pnl"],
        "gross_pnl": r["gross_pnl"],
        "position": r["positions"],
    })
    df["month"] = df["timestamp"].dt.to_period("M")
    monthly = df.groupby("month").agg(
        net_pnl=("net_pnl", "sum"),
        gross_pnl=("gross_pnl", "sum"),
        avg_pos=("position", "mean"),
        abs_pos=("position", lambda x: np.abs(x).mean()),
        n=("net_pnl", "count"),
    )
    monthly["cost"] = monthly["gross_pnl"] - monthly["net_pnl"]

    print(f"\n{'='*70}")
    print(f"  {split_name.upper()} Monthly PnL (bps)")
    print(f"{'='*70}")
    print(monthly.to_string(float_format="{:+.0f}".format))
    print(f"\n  Total net: {monthly['net_pnl'].sum():+,.0f} bps")
    print(f"  Total cost: {monthly['cost'].sum():+,.0f} bps")